This is where MacroSense learns. We now feed our 
carefully engineered features into four machine learning models 
and train each one to recognise the patterns that connect current 
economic conditions to future outcomes.

We build four models deliberately rather than just one. No single 
model is best for every situation. 
Each model brings a different strength and together they form 
the foundation of our ensemble, a combined forecast that is 
typically more accurate and more robust than any individual model.

The four models we build are Linear Regression as our baseline, 
Ridge Regression as a regularised improvement, Random Forest as 
our non linear tree based model, and XGBoost as our primary and 
most powerful model.

Input: data/featured_fred_data.csv
Output: outputs/ (all trained models saved here)

Load our featured dataset and build the train test 
split.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [4]:
os.makedirs('../outputs', exist_ok=True)
fe_clean = pd.read_csv('../data/featured_fred_data.csv',
                        index_col=0,
                        parse_dates=True)

target_cols  = ['target_GDP', 
                'target_Inflation', 
                'target_Unemployment']

feature_cols = [c for c in fe_clean.columns 
                if c not in target_cols]

X = fe_clean[feature_cols]
y = fe_clean[target_cols]

split_point    = int(len(fe_clean) * 0.8)
X_train        = X.iloc[:split_point]
X_test         = X.iloc[split_point:]
y_train        = y.iloc[:split_point]
y_test         = y.iloc[split_point:]

scaler         = joblib.load('../outputs/scaler.pkl')
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Setup complete")
print(f"Training: {X_train.shape[0]} months")
print(f"Testing:  {X_test.shape[0]} months")
print(f"Features: {X_train.shape[1]}")

Setup complete
Training: 305 months
Testing:  77 months
Features: 74


Here I Build a reusable training function. Rather than writing 
separate training code for each model and each target we write 
one function that handles any model and any target. This keeps 
our code clean, consistent, and easy to debug.

In [5]:
def train_model(model, X_tr, y_tr, model_name, target_name):
    """
    Trains a model on the given data and saves it to disk.
    Returns the trained model.
    """
    print(f"  Training {model_name} for {target_name}...", end=' ')
    
    model.fit(X_tr, y_tr)
    
    filename = f'../outputs/{model_name}_{target_name}.pkl'
    joblib.dump(model, filename)
    
    print("done")
    return model

print("Training function defined successfully")

Training function defined successfully


In [6]:
print("MODEL 1: LINEAR REGRESSION")
lr_models = {}

for target in target_cols:
    lr_models[target] = train_model(
        LinearRegression(),
        X_train_scaled,
        y_train[target],
        'LinearRegression',
        target
    )

print("\nAll Linear Regression models trained and saved")

MODEL 1: LINEAR REGRESSION
  Training LinearRegression for target_GDP... done
  Training LinearRegression for target_Inflation... done
  Training LinearRegression for target_Unemployment... done

All Linear Regression models trained and saved


In [7]:
print("MODEL 2: RIDGE REGRESSION")
ridge_models = {}

for target in target_cols:
    ridge_models[target] = train_model(
        Ridge(alpha=1.0),
        X_train_scaled,
        y_train[target],
        'Ridge',
        target
    )

print("\nAll Ridge Regression models trained and saved")

MODEL 2: RIDGE REGRESSION
  Training Ridge for target_GDP... done
  Training Ridge for target_Inflation... done
  Training Ridge for target_Unemployment... done

All Ridge Regression models trained and saved


In [8]:
print("MODEL 3: RANDOM FOREST")
rf_models = {}

for target in target_cols:
    rf_models[target] = train_model(
        RandomForestRegressor(
            n_estimators=200,
            max_depth=6,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ),
        X_train,
        y_train[target],
        'RandomForest',
        target
    )

print("\nAll Random Forest models trained and saved")

MODEL 3: RANDOM FOREST
  Training RandomForest for target_GDP... done
  Training RandomForest for target_Inflation... done
  Training RandomForest for target_Unemployment... done

All Random Forest models trained and saved


In [9]:
print("MODEL 4: XGBOOST (Primary Model)")
xgb_models = {}

for target in target_cols:
    xgb_models[target] = train_model(
        XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0
        ),
        X_train,
        y_train[target],
        'XGBoost',
        target
    )

print("\nAll XGBoost models trained and saved")

MODEL 4: XGBOOST (Primary Model)
  Training XGBoost for target_GDP... done
  Training XGBoost for target_Inflation... done
  Training XGBoost for target_Unemployment... done

All XGBoost models trained and saved


In [10]:
print("GENERATING ALL PREDICTIONS")
predictions = {}

for target in target_cols:
    
    pred_lr    = lr_models[target].predict(X_test_scaled)
    pred_ridge = ridge_models[target].predict(X_test_scaled)
    pred_rf    = rf_models[target].predict(X_test)
    pred_xgb   = xgb_models[target].predict(X_test)
    
    pred_ensemble = (pred_lr + pred_ridge + pred_rf + pred_xgb) / 4
    
    predictions[target] = {
        'LinearRegression' : pred_lr,
        'Ridge'            : pred_ridge,
        'RandomForest'     : pred_rf,
        'XGBoost'          : pred_xgb,
        'Ensemble'         : pred_ensemble
    }
    
    print(f"Predictions generated for {target}")

joblib.dump(predictions, '../outputs/all_predictions.pkl')
print("\nAll predictions saved to outputs/all_predictions.pkl")

GENERATING ALL PREDICTIONS
Predictions generated for target_GDP
Predictions generated for target_Inflation
Predictions generated for target_Unemployment

All predictions saved to outputs/all_predictions.pkl


In [13]:
print("Preview of predictions for target_GDP:")
print(f"{'Model':<20} {'First Pred':>12} {'Last Pred':>12} {'Mean':>12}")

for model_name, preds in predictions['target_GDP'].items():
    print(f"{model_name:<20} {preds[0]:>12.3f} {preds[-1]:>12.3f} {preds.mean():>12.3f}")

print(f"\nActual GDP growth (test set):")
print(f"  Mean:  {y_test['target_GDP'].mean():.3f}%")
print(f"  Min:   {y_test['target_GDP'].min():.3f}%")
print(f"  Max:   {y_test['target_GDP'].max():.3f}%")

Preview of predictions for target_GDP:
Model                  First Pred    Last Pred         Mean
LinearRegression            1.020       -0.248        0.876
Ridge                       1.006       -0.254        0.788
RandomForest                0.647        0.884        0.520
XGBoost                     0.645        1.083        0.713
Ensemble                    0.830        0.367        0.724

Actual GDP growth (test set):
  Mean:  0.786%
  Min:   -9.089%
  Max:   8.984%


In [14]:
print("Checking saved model files:")
print()

model_files = [f for f in os.listdir('../outputs') if f.endswith('.pkl')]
model_files.sort()

for f in model_files:
    filepath = os.path.join('../outputs', f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f:<45} {size_kb:.1f} KB")

print(f"\nTotal files saved: {len(model_files)}")

Checking saved model files:

  LinearRegression_target_GDP.pkl               1.7 KB
  LinearRegression_target_Inflation.pkl         1.7 KB
  LinearRegression_target_Unemployment.pkl      1.7 KB
  RandomForest_target_GDP.pkl                   554.0 KB
  RandomForest_target_Inflation.pkl             616.3 KB
  RandomForest_target_Unemployment.pkl          734.5 KB
  Ridge_target_GDP.pkl                          1.1 KB
  Ridge_target_Inflation.pkl                    1.1 KB
  Ridge_target_Unemployment.pkl                 1.1 KB
  XGBoost_target_GDP.pkl                        450.4 KB
  XGBoost_target_Inflation.pkl                  456.0 KB
  XGBoost_target_Unemployment.pkl               458.3 KB
  all_predictions.pkl                           9.3 KB
  scaler.pkl                                    4.0 KB

Total files saved: 14


Four models have been trained on 25 years 
of monthly US economic data and saved to disk. Each model learned 
a different mathematical representation of how past economic 
conditions connect to future GDP growth, inflation, and unemployment.

What happens next is where we find out how well 
they actually learned. Training performance can be misleading; 
A model can memorise training data perfectly and still fail 
completely on new data. Evaluation on the held-out test set is 
the moment of truth.

Models saved:
LinearRegression x 3 targets
Ridge x 3 targets  
RandomForest x 3 targets
XGBoost x 3 targets
Ensemble predictions x 3 targets

Total: 12 trained models plus ensemble predictions